<a href="https://colab.research.google.com/github/rezve66/rice-trait-network-yield-drivers-/blob/main/genotype_trait_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🌾 Integrative Genotype–Trait Multi‑Analysis Pipeline
### BLUP Heritability • Trait Networks • Yield Drivers • MGIDI • Machine Learning

**Author:** Jarvis  
**Affiliation:** Plant Protection / Plant Breeding & Biotechnology Research  
**Purpose:** Reproducible multi‑omics‑compatible phenotypic analysis workflow for crop improvement research.

---

## 🎯 Scientific Objective

This notebook implements a **complete phenotype analysis pipeline** commonly used in quantitative genetics and crop breeding studies.

The workflow integrates:

1. **BLUP estimation using mixed models (REML)**
2. **Variance component estimation**
3. **BLUP‑based heritability**
4. **Partial correlation analysis (Graphical Lasso)**
5. **Trait interaction networks**
6. **Yield driver modeling**
7. **PLS‑VIP trait importance**
8. **Machine learning feature importance**
9. **MGIDI multi‑trait selection index**
10. **Publication‑ready figure generation (600 dpi)**

---

## 📊 Expected Input Dataset Format

The dataset should be in **Excel (.xlsx) or CSV (.csv)** format.

Example structure:

| Genotype | Replication | Days to flowering | Plant height | Panicle length | Grain yield per hill |
|----------|-------------|------------------|-------------|---------------|---------------------|
| G1 | R1 | 82 | 115 | 25.2 | 28.4 |
| G1 | R2 | 81 | 117 | 24.8 | 27.9 |
| G2 | R1 | 79 | 120 | 26.1 | 29.5 |

Required columns:

• **Genotype**  
• **Replication**  
• Trait columns

---

## 📁 Recommended GitHub Repository Structure

```
Genotype-Trait-MultiAnalysis
│
├── notebooks
│   └── genotype_trait_analysis.ipynb
│
├── data
│   └── example_dataset.xlsx
│
├── outputs
│   ├── figures
│   └── tables
│
├── README.md
└── requirements.txt
```

---


## 1️⃣ Install Required Python Packages

In [ ]:

!pip -q install statsmodels scikit-learn networkx seaborn shap openpyxl


## 2️⃣ Import Libraries

In [ ]:

import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

import statsmodels.formula.api as smf

from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLassoCV
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score
from sklearn.decomposition import FactorAnalysis

warnings.filterwarnings("ignore")



## 3️⃣ Upload Dataset

Upload your dataset directly in Google Colab.
Supported formats:

• `.xlsx`  
• `.csv`


In [ ]:

from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith(".xlsx"):
    df = pd.read_excel(filename)
else:
    df = pd.read_csv(filename)

df.columns = df.columns.str.strip()

print("Dataset loaded successfully")
print(df.head())



## 4️⃣ Detect Trait Columns

Automatically identify all phenotype traits.


In [ ]:

traits = [c for c in df.columns if c not in ["Genotype","Replication"]]

print("Traits detected:", len(traits))
print(traits)



## 5️⃣ BLUP Estimation (Mixed Linear Model)

Statistical model:

**yᵢⱼ = μ + Rⱼ + Gᵢ + εᵢⱼ**

Where

μ = overall mean  
Rⱼ = replication effect (fixed)  
Gᵢ = genotype effect (random)  
εᵢⱼ = residual error

BLUP values represent **genotype performance adjusted for replication effects**.


In [ ]:

genotypes = sorted(df["Genotype"].astype(str).unique())

blup = pd.DataFrame(index=genotypes, columns=traits)

for t in traits:
    formula = f'Q("{t}") ~ C(Replication)'

    md = smf.mixedlm(formula, df, groups=df["Genotype"])
    m = md.fit(reml=True)

    mu = float(m.fe_params.iloc[0])

    for g in genotypes:
        u = float(np.asarray(m.random_effects[g])[0])
        blup.loc[g, t] = mu + u

blup.head()


## 6️⃣ Standardize BLUP Matrix (Z‑score scaling)

In [ ]:

Z = pd.DataFrame(
    StandardScaler().fit_transform(blup.values),
    columns=traits,
    index=blup.index
)

Z.head()



## 7️⃣ Partial Correlation Analysis (Graphical Lasso)

This method estimates **conditional relationships among traits**.


In [ ]:

gl = GraphicalLassoCV()
gl.fit(Z.values)

P = gl.precision_
d = np.sqrt(np.diag(P))

pcorr = -P / np.outer(d, d)
np.fill_diagonal(pcorr, 1)

pcorr_df = pd.DataFrame(pcorr, index=traits, columns=traits)


In [ ]:

plt.figure(figsize=(10,8))
sns.heatmap(pcorr_df, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("BLUP‑based Partial Correlation Matrix")
plt.show()



## 8️⃣ Trait Interaction Network


In [ ]:

THRESHOLD = 0.20

G = nx.Graph()

for i in range(len(traits)):
    for j in range(i+1,len(traits)):
        r = pcorr_df.iloc[i,j]
        if abs(r) >= THRESHOLD:
            G.add_edge(traits[i], traits[j], weight=r)

plt.figure(figsize=(10,10))

pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=2000)

plt.title("Trait Interaction Network")
plt.show()



## 9️⃣ Yield Driver Identification

Multiple Linear Regression is used to detect traits that most strongly influence yield.


In [ ]:

yield_trait = "Grain yield per hill"

X = Z.drop(columns=[yield_trait])
y = Z[yield_trait]

model = LinearRegression().fit(X,y)

importance = pd.Series(abs(model.coef_), index=X.columns).sort_values()

importance.plot.barh()
plt.title("Yield Driver Traits")
plt.show()



## 🔟 Random Forest Feature Importance


In [ ]:

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X,y)

imp = pd.Series(rf.feature_importances_, index=X.columns)

imp.sort_values().plot.barh()
plt.title("Random Forest Trait Importance")
plt.show()



## 1️⃣1️⃣ MGIDI Multi‑Trait Selection Index


In [ ]:

gmeans = df.groupby("Genotype")[traits].mean()

Z = StandardScaler().fit_transform(gmeans)

fa = FactorAnalysis(n_components=min(5, Z.shape[1]))
F = fa.fit_transform(Z)

ideotype = F.max(axis=0)

mgidi = np.sqrt(((F - ideotype)**2).sum(axis=1))
mgidi = pd.Series(mgidi, index=gmeans.index).sort_values()

mgidi.head()



## 1️⃣2️⃣ Export Results


In [ ]:

os.makedirs("analysis_outputs", exist_ok=True)

blup.to_csv("analysis_outputs/blup_matrix.csv")
pcorr_df.to_csv("analysis_outputs/partial_correlation_matrix.csv")
mgidi.to_csv("analysis_outputs/mgidi_scores.csv")

print("All outputs exported.")



# ✅ Pipeline Completed

Generated outputs:

• BLUP matrix  
• Partial correlation matrix  
• Trait interaction network  
• Yield predictor model  
• Machine learning importance  
• MGIDI selection index  

These outputs can directly support **scientific publications and PhD research portfolios**.
